# Lesson 1: Your First LangGraph — Hardcoded Sum Node

## Objective
Build the absolute minimum LangGraph: one node, one operation, one result.  
Understand the core building blocks before adding any complexity.

## Problem Statement
Before using LangGraph in production you must understand:
- What is a **graph** in LangGraph?
- What is a **node**?
- What is **state**?
- How does execution flow `START → node → END`?

## Theory: What is LangGraph?

LangGraph is a library built on top of LangChain for building **stateful, multi-step AI workflows** modeled as directed graphs.

Think of it as a programmable flowchart where:
| Concept | Meaning |
|---------|---------|
| **Node** | A Python function that does work and updates state |
| **Edge** | A connection that controls which node runs next |
| **State** | A shared dictionary passed through every node |
| **Graph** | The container that wires nodes + edges together |

### The Pregel Execution Model
LangGraph uses **Pregel-style** execution (named after Google's distributed graph system):
1. All active nodes receive the **current state**
2. Each node computes and returns a **partial update** (only changed fields)
3. LangGraph **merges** the partial update into a new state snapshot
4. Execution advances to the next node(s)

This is fundamentally different from a simple function pipeline:
- State is **immutable** — each step creates a new version
- You can **pause** mid-execution (checkpointing — Lesson 17)
- You can **branch** based on state values (routing — Lesson 3)
- You can **loop** by pointing edges backward (Lesson 5)
- Multiple nodes can run **in parallel** (Lesson 9)

## What is NEW in Lesson 1?
This is the foundation. Every concept here is used in all future lessons:
- ✅ `StateGraph` — the graph container
- ✅ `START` / `END` — special entry and exit sentinels
- ✅ `add_node()` — register a Python function as a graph node
- ✅ `add_edge()` — connect two nodes
- ✅ `compile()` — lock the graph into an executable
- ✅ `invoke()` — run the graph synchronously


In [ ]:
# ── Cell 1: Imports ─────────────────────────────────────────────────────────
# StateGraph  → the main class for building a graph
# START       → special sentinel: the entry point of the graph
# END         → special sentinel: the exit point of the graph
# TypedDict   → Python type for defining structured state schemas

from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from IPython.display import Image, display

print("✓ Imports successful")
print(f"  StateGraph: {StateGraph}")
print(f"  START sentinel: {START!r}")
print(f"  END sentinel:   {END!r}")


## Step 1: Define the State Schema

State is a `TypedDict` — a Python dictionary with type annotations.

**Every node** in the graph:
- Receives the full current state as input
- Returns a **partial dict** with only the fields it modified

LangGraph merges the partial dict back into the state automatically.


In [ ]:
# ── Cell 2: State Schema ────────────────────────────────────────────────────
# TypedDict gives us a typed dictionary schema
# LangGraph reads this to know which fields are legal
#
# Rule: Keep state minimal — only the fields ALL nodes might need

class State(TypedDict):
    a: int        # First number (input)
    b: int        # Second number (input)
    result: int   # Sum of a + b (output)

print("✓ State schema defined")
print("  Fields:", list(State.__annotations__.keys()))
print("  Types: ", {k: v.__name__ for k, v in State.__annotations__.items()})


## Step 2: Define the Node Function

A **node** is a plain Python function that:
1. Takes `state: State` as input (the full current state dict)
2. Does work (computation, LLM call, API request, etc.)
3. Returns a `dict` with **only the fields it wants to update**

⚠️ Critical rule: **return only what changed**.  
LangGraph merges your return value into the current state — untouched fields remain as-is.


In [ ]:
# ── Cell 3: Node Function ───────────────────────────────────────────────────
# The node receives the entire state dict
# It returns ONLY the fields it modifies — this is a "partial update"

def add_node(state: State) -> dict:
    print(f"  [add_node] Received state: a={state['a']}, b={state['b']}")
    
    result = state["a"] + state["b"]    # The actual work
    
    print(f"  [add_node] Computed: {state['a']} + {state['b']} = {result}")
    
    return {"result": result}           # Partial update — only 'result' changes
    #              ↑
    # LangGraph merges this into state: {a:7, b:3, result:10}
    # Fields 'a' and 'b' are NOT in the return → they stay unchanged

print("✓ Node function defined")


## Step 3: Build & Compile the Graph

The builder pattern:
1. `StateGraph(State)` — create a graph with your schema
2. `add_node(name, fn)` — register each function
3. `add_edge(from, to)` — connect nodes
4. `compile()` — validate and lock the graph

After `compile()`, the graph becomes a **LangChain Runnable** (supports `.invoke()`, `.stream()`, `.batch()`).


In [ ]:
# ── Cell 4: Build the Graph ─────────────────────────────────────────────────

builder = StateGraph(State)         # Create graph with our schema

# Register nodes
# Syntax: add_node("unique_name", function)
# The name is used when creating edges
builder.add_node("add_node", add_node)

# Create edges
# START → add_node  (where execution enters the graph)
builder.add_edge(START, "add_node")

# add_node → END    (after add_node, execution exits)
builder.add_edge("add_node", END)

# Compile: validates structure, returns an executable CompiledGraph
graph = builder.compile()

print("✓ Graph compiled successfully")
print(f"  Graph type: {type(graph)}")
print(f"  Nodes: {list(graph.get_graph().nodes.keys())}")


## Step 4: Visualize the Graph

LangGraph can render itself as a **Mermaid diagram** — this is essential for debugging complex graphs.  
Always visualize before running to confirm the structure is what you intended.


In [ ]:
# ── Cell 5: Visualize ───────────────────────────────────────────────────────
# draw_mermaid_png() renders the graph as a PNG image
# You should see:  __start__ → add_node → __end__

display(Image(graph.get_graph().draw_mermaid_png()))


## Step 5: Run the Graph

`invoke(initial_state)`:
- Accepts a dict matching your State schema
- Runs all nodes in order
- Returns the **final state** (all fields, including unchanged ones)


In [ ]:
# ── Cell 6: Run the Graph ───────────────────────────────────────────────────
# invoke() is synchronous — it blocks until the graph finishes
# Returns the complete final state (all fields)

initial_state = {
    "a": 7,
    "b": 3,
    "result": 0    # placeholder — will be overwritten by add_node
}

print("Running graph...")
print("-" * 40)

final_state = graph.invoke(initial_state)

print("-" * 40)
print(f"\nInput  : a = {final_state['a']}, b = {final_state['b']}")
print(f"Output : result = {final_state['result']}")
print(f"\nFull final state: {final_state}")


In [ ]:
# ── Cell 7: More Examples ───────────────────────────────────────────────────

print("Testing multiple inputs:\n")
test_cases = [
    {"a": 10,  "b": 5,   "result": 0},
    {"a": 100, "b": 200, "result": 0},
    {"a": 0,   "b": 42,  "result": 0},
    {"a": -5,  "b": 15,  "result": 0},
]

for case in test_cases:
    out = graph.invoke(case)
    print(f"  {out['a']:>4} + {out['b']:>4} = {out['result']}")


## Code Explanation — Every Line

| Line | Purpose |
|------|---------|
| `StateGraph(State)` | Creates the graph container; State schema defines valid fields |
| `add_node("add_node", add_node)` | Registers the function; first arg is the name used in edges |
| `add_edge(START, "add_node")` | Execution entry: first node to run |
| `add_edge("add_node", END)` | Execution exit: graph finishes after add_node |
| `compile()` | Validates graph (checks for disconnected nodes, missing edges), returns Runnable |
| `invoke(state)` | Runs the graph synchronously, returns final state |
| `return {"result": result}` | **Partial update** — only updates the `result` field |

## Summary

You built a minimal but complete LangGraph:
- Accepted initial state `{a, b, result}`
- Ran `add_node` which computed `a + b`
- Returned the updated state with `result` filled in

## Why This Matters in Production

Even this trivial example demonstrates the universal LangGraph pattern used in production:

```
state in → [node 1] → [node 2] → ... → [node N] → state out
```

Swapping `add_node` for an LLM call, API request, or database query gives you a production agent.

## Limitations → What Lesson 2 Solves
- ❌ Only one operation (hardcoded add)
- ❌ No way to run different operations
- ❌ **Lesson 2**: Multiple nodes in sequence with a richer state schema
